# Module 1 lab: the quantum Fourier transform and phase estimation

**Quantum Computing Intermediate · Module 1 · about 90 minutes** · notebook version 2026-10-05b

The quantum Fourier transform (QFT) turns a pattern of **phases** into a number you can measure. Phase estimation uses it to read the eigenphase of a gate into a register of qubits. Both are building blocks of Shor's algorithm (Module 3) and of many chemistry algorithms.

In this lab you:

1. find eigenvalues and eigenphases of gates with NumPy;
2. build the 3-qubit QFT gate by gate, `qft3()`, and check it against the discrete Fourier transform (DFT) matrix;
3. write the QFT for any number of qubits, `qft(n)`;
4. write `phase_estimation()` and use it on the T gate (phase 1/8) and on a phase of 1/3;
5. get your verification value.

The **Module 1 lab check** in Canvas asks for results from this notebook, so keep it open in a second tab. Run each code cell with **Shift + Enter**, in order.

## Step 0: start Python

Run the cell below. The first time, Python can take up to a minute to start in your browser. It also prepares two helpers: `dft_matrix(n)`, the 2ⁿ × 2ⁿ DFT matrix, and `show_amplitudes(state)`, which prints each amplitude's size and phase.

In [ ]:
import math
import numpy as np
import qsim
from qsim import QuantumCircuit, Statevector, Operator, AerSimulator, plot_histogram

sim = AerSimulator(seed_simulator=7)


def dft_matrix(n):
    """Prepared for you: the 2^n x 2^n DFT matrix, entry (k, j) = exp(2 pi i j k / 2^n) / sqrt(2^n)."""
    N = 2 ** n
    w = np.exp(2j * np.pi / N)
    return np.array([[w ** (j * k) for j in range(N)] for k in range(N)]) / math.sqrt(N)


def show_amplitudes(state):
    """Prepared for you: each basis state's amplitude as a size and a phase in degrees (0 to 360)."""
    v = np.asarray(state)
    n = int(round(math.log2(len(v))))
    for k, a in enumerate(v):
        if abs(a) > 1e-9:
            deg = (math.degrees(math.atan2(a.imag, a.real)) + 360) % 360
            print(f"|{format(k, f'0{n}b')}>  (k = {k}):  size {abs(a):.4f},  phase {round(deg, 2) % 360:6.1f} degrees")


print("Ready. qsim", qsim.__version__, "and NumPy", np.__version__)

## Step 1: eigenvalues and eigenphases

An **eigenvector** of a gate U is a state that U leaves unchanged except for a factor: U|ψ⟩ = λ|ψ⟩. For a unitary gate every eigenvalue λ has size 1, so it can be written as λ = e^(2πiφ) with φ between 0 and 1. That number φ is the **eigenphase**, and phase estimation measures it.

The T gate is diag(1, e^(iπ/4)). Its eigenvectors are |0⟩ and |1⟩, with eigenphases 0 and 1/8. Run the cell: NumPy finds the eigenvalues, and the cell converts each to an eigenphase.

In [ ]:
def eigenphases(U):
    """The eigenphases phi (0 <= phi < 1) of a unitary matrix, from its eigenvalues exp(2 pi i phi)."""
    values = np.linalg.eigvals(np.asarray(U))
    return sorted(float(round((np.angle(v) / (2 * math.pi)) % 1, 6)) for v in values)


T = np.diag([1, np.exp(1j * math.pi / 4)])
P_third = np.diag([1, np.exp(2j * math.pi / 3)])     # the phase gate P(2 pi / 3)
print("T gate eigenphases:      ", eigenphases(T))
print("P(2 pi / 3) eigenphases: ", eigenphases(P_third))

**Your turn (not graded).** In the next cell, find the eigenphases of the S gate, diag(1, i), and of the Hadamard gate. For H, which eigenphase does the eigenvalue −1 give?

In [ ]:
# Your turn: eigenphases of S and of H


## Step 2: the 3-qubit QFT, gate by gate

The QFT on n qubits does to a state's amplitudes what the DFT matrix does to a vector: a basis state |j⟩ becomes

  QFT|j⟩ = (1/√2ⁿ) Σₖ e^(2πi·j·k/2ⁿ) |k⟩,

an equal superposition in which the phase of |k⟩ grows in steps of j/2ⁿ of a full turn.

The circuit works on the highest qubit first. For 3 qubits, with Qiskit's order (qubit 0 on the right):

1. H on qubit 2, then a controlled-phase `cp(pi/2, 1, 2)` and `cp(pi/4, 0, 2)`;
2. H on qubit 1, then `cp(pi/2, 0, 1)`;
3. H on qubit 0;
4. swap qubits 0 and 2, so the output comes out in the right order.

The angle of the controlled phase between qubits k and j is π/2^(j − k). **Your task:** write `qft3()` that returns this 3-qubit circuit. Build it from gates; do not use a matrix.

In [ ]:
def qft3():
    """Return the 3-qubit QFT as a QuantumCircuit built from h, cp and swap gates."""
    qc = QuantumCircuit(3)
    # YOUR CODE HERE
    raise NotImplementedError("Complete qft3() first.")
    return qc

Check it against the DFT matrix. `Operator(circuit)` gives the circuit's 8 × 8 matrix.

In [ ]:
diff = np.abs(Operator(qft3()).data - dft_matrix(3)).max()
print(qft3().draw())
print("largest difference from the DFT matrix:", f"{diff:.1e}")
print("qft3() matches the DFT" if diff < 1e-8 else "Not yet: compare your angles and the final swap with the steps above.")

## Step 3: what the QFT does to a basis state

Prepare |3⟩ = |011⟩ (X on qubits 0 and 1), apply your `qft3()`, and look at the amplitudes. Every amplitude has size 1/√8 ≈ 0.354; the phases step by 3/8 of a full turn (135°) from one k to the next. The lab check asks for one of these phases.

In [ ]:
qc = QuantumCircuit(3)
qc.x(0)
qc.x(1)
qc.compose(qft3(), inplace=True)
state = Statevector(qc)
show_amplitudes(state)
print()
print("phase of |010> (k = 2):", round((math.degrees(np.angle(state[2])) + 360) % 360, 1), "degrees")

## Step 4: the QFT for any number of qubits

The same pattern works for n qubits: for each qubit j from n − 1 down to 0, an H on qubit j, then `cp(pi / 2**(j - k), k, j)` for every k below j; at the end, swap qubit i with qubit n − 1 − i for i = 0 up to n // 2 − 1.

**Your task:** complete `qft(n)`. The two loops and the swap loop are written for you; fill in the three gates marked `YOUR CODE HERE`.

In [ ]:
def qft(n):
    """Return the n-qubit QFT as a QuantumCircuit."""
    qc = QuantumCircuit(n)
    for j in reversed(range(n)):
        # YOUR CODE HERE: an H gate on qubit j
        for k in reversed(range(j)):
            # YOUR CODE HERE: the controlled phase between qubits k and j, angle pi / 2**(j - k)
            pass
    for i in range(n // 2):
        # YOUR CODE HERE: swap qubit i with qubit n - 1 - i
        pass
    if not qc.data:    # leave this line: it stops the lab until you have added gates
        raise NotImplementedError("Complete qft(n) first: fill in the three gates marked YOUR CODE HERE.")
    return qc

In [ ]:
ok = True
for n in range(1, 6):
    c = qft(n)
    diff = np.abs(Operator(c).data - dft_matrix(n)).max()
    names = [i.name for i in c.data]
    print(f"n = {n}: largest difference {diff:.1e};  {names.count('h')} H, {names.count('cp')} controlled-phase, "
          f"{names.count('swap')} swap gates")
    ok = ok and diff < 1e-8
print("qft(n) matches the DFT for n = 1 to 5" if ok else "Not yet: compare your gates with the pattern above.")

**What to notice.** The number of controlled-phase gates is n(n − 1)/2, so the QFT needs about n²/2 gates. The classical fast Fourier transform needs about n·2ⁿ operations on 2ⁿ numbers. The QFT is exponentially smaller, but its output is a quantum state: you cannot read all 2ⁿ amplitudes, only measure it. Algorithms use it where one measurement is enough, as in phase estimation.

## Step 5: phase estimation

Phase estimation finds the eigenphase φ of a gate U for one of its eigenvectors. Here U is the phase gate P(2πφ) = diag(1, e^(2πiφ)), and its eigenvector |1⟩ has eigenphase φ. The circuit has t **counting qubits** (0 to t − 1) and one **target qubit** (qubit t):

1. Put the target in the eigenvector |1⟩ (an X gate) and each counting qubit in |+⟩ (H gates).
2. Counting qubit k controls U raised to the power 2ᵏ. For the phase gate, U^(2ᵏ) is P(2π·φ·2ᵏ), so this is one `cp(2 * math.pi * phase * 2**k, k, t)`. By phase kickback (Level 1, Module 6), counting qubit k picks up the phase e^(2πi·φ·2ᵏ) on its |1⟩ part.
3. The counting register now holds (1/√2ᵗ) Σⱼ e^(2πi·φ·j)|j⟩. If x = φ·2ᵗ is a whole number, this is exactly the QFT of |x⟩, so the inverse QFT turns it into |x⟩ and every shot gives x (Step 6). If φ·2ᵗ is not a whole number, the state is not the QFT of any single basis state, and the inverse QFT gives a spread of results that peaks at the closest whole numbers (Step 7). In both cases, apply the **inverse** QFT, `qft(t).inverse()`, to the counting qubits. (`.inverse()` applies the inverse of every gate in reverse order.)
4. Measure counting qubit k into classical bit k. The result m, read as a binary number, gives the estimate φ ≈ m / 2ᵗ.

**Your task:** write `phase_estimation(phase, t)` following these four steps. Use `qc.compose(qft(t).inverse(), qubits=list(range(t)), inplace=True)` for step 3.

In [ ]:
def phase_estimation(phase, t):
    """Phase estimation of P(2 pi phase) on its eigenvector |1>: t counting qubits, target qubit t."""
    qc = QuantumCircuit(t + 1, t)
    # YOUR CODE HERE
    raise NotImplementedError("Complete phase_estimation() first.")
    return qc

## Step 6: the T gate, an exact case

The T gate's eigenphase is 1/8 = 0.001 in binary, which fits exactly in 3 bits. With 3 counting qubits, every shot should give the result 001, so the estimate is exactly 1/8. The lab check asks for the estimate.

In [ ]:
qc = phase_estimation(1 / 8, 3)
print(qc.draw())
counts = sim.run(qc, shots=1000).result().get_counts()
print(counts)
m = int(max(counts, key=counts.get), 2)
print("most frequent result:", format(m, "03b"), "= m =", m, "  estimate m / 8 =", m / 8)

## Step 7: a phase of 1/3, which no number of bits can hold exactly

1/3 = 0.010101... in binary never ends, so no result is exactly right. Phase estimation then gives the closest t-bit fraction with probability at least 4/π² ≈ 0.405; the remaining probability is spread over the other t-bit fractions, mostly the ones next to it. The closest fraction is never more than 1/2^(t+1) from φ, but that bound is for the best estimate, not for every shot: a single shot can return a fraction farther away. Each extra counting qubit halves the gap between neighbouring estimates.

The cell computes the exact probabilities for t = 3 to 6 (with `Statevector(...).probabilities(qargs)`, new in qsim 1.6.0, which measures only the counting qubits) and draws the histogram for t = 5. The lab check asks for the estimate with 6 counting qubits.

In [ ]:
for t in range(3, 7):
    qc = phase_estimation(1 / 3, t)
    qc.remove_final_measurements(inplace=True)
    p = Statevector(qc).probabilities(list(range(t)))
    m = int(np.argmax(p))
    print(f"t = {t}: most likely m = {m:2d} of {2 ** t},  estimate {m / 2 ** t:.6f},  error {abs(m / 2 ** t - 1 / 3):.6f},  "
          f"probability {p[m]:.4f},  largest possible error of the best estimate 1/2^(t+1) = {1 / 2 ** (t + 1):.6f}")

counts = sim.run(phase_estimation(1 / 3, 5), shots=1000).result().get_counts()
plot_histogram(counts, title="phase 1/3, 5 counting qubits, 1,000 shots")

## Step 8: your personal check

Open the **Module 1 lab check** in Canvas. Question 1 shows your own phase PHI and seed SEED. Type them below and run the next two cells. The check cell tests `qft3()`, `qft(n)` for n = 1 to 5, and `phase_estimation()` on seven phases. Only if every test passes does it print your **verification value**: the number of shots, out of 1,000, in which the course's reference phase-estimation circuit with 5 counting qubits returns the best 5-bit estimate of PHI, run with your seed. Type it into Canvas.

In [ ]:
PHI = 0.0     # your phase from Canvas, for example 0.432
SEED = 0      # your seed from Canvas, for example 512

In [ ]:
import math

import numpy as np
import qsim
from qsim import AerSimulator, Operator, QuantumCircuit


def dft_matrix(n):
    """The 2^n x 2^n discrete Fourier transform matrix: entry (k, j) = exp(2 pi i j k / 2^n) / sqrt(2^n)."""
    N = 2 ** n
    w = np.exp(2j * np.pi / N)
    return np.array([[w ** (j * k) for j in range(N)] for k in range(N)]) / math.sqrt(N)


def reference_qft(n):
    qc = QuantumCircuit(n)
    for j in reversed(range(n)):
        qc.h(j)
        for k in reversed(range(j)):
            qc.cp(math.pi / 2 ** (j - k), k, j)
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)
    return qc


def reference_phase_estimation(phase, t):
    qc = QuantumCircuit(t + 1, t)
    qc.x(t)
    for k in range(t):
        qc.h(k)
    for k in range(t):
        qc.cp(2 * math.pi * phase * 2 ** k, k, t)
    qc.compose(reference_qft(t).inverse(), qubits=list(range(t)), inplace=True)
    qc.measure(list(range(t)), list(range(t)))
    return qc


def exact_distribution(qc):
    """The exact probability of each result string of a circuit (qsim's branch simulator)."""
    out = {}
    for rho, bits in qsim._branch_states(qc, None):
        w = float(np.real(np.trace(rho)))
        if w > 1e-12:
            key = qsim._format_key(list(bits), qc._cregs)
            out[key] = out.get(key, 0.0) + w
    return out


def personal_value(phi, seed):
    t = 5
    best = format(round(phi * 2 ** t) % 2 ** t, f"0{t}b")
    counts = AerSimulator(seed_simulator=int(seed)).run(reference_phase_estimation(float(phi), t), shots=1000).result().get_counts()
    return int(counts.get(best, 0))


def _bit_reversal(n):
    N = 2 ** n
    perm = np.zeros((N, N))
    for i in range(N):
        perm[int(format(i, f"0{n}b")[::-1], 2), i] = 1
    return perm


def _diagnose_qft(m, n):
    """A hint for a wrong n-qubit QFT matrix m."""
    F = dft_matrix(n)
    if np.allclose(m, F.conj(), atol=1e-8):
        return "it is the inverse QFT: the controlled-phase angles need the opposite sign."
    if n > 1 and (np.allclose(m, _bit_reversal(n) @ F, atol=1e-8) or np.allclose(m, F @ _bit_reversal(n), atol=1e-8)):
        return "the qubit order is reversed: add the swaps at the end (qubit i with qubit n - 1 - i)."
    if Operator(m).equiv(Operator(F)):
        return "it matches the DFT only up to a global phase; the QFT should match it exactly."
    return "compare each controlled-phase angle with pi / 2^(j - k), and check that every qubit gets its H gate."


def _test_qft3(qft3):
    try:
        qc = qft3()
    except NotImplementedError:
        return False, ["qft3() is not written yet."]
    except Exception as e:  # noqa: BLE001
        return False, [f"qft3() raised {type(e).__name__}: {e}"]
    if not isinstance(qc, QuantumCircuit) or qc.num_qubits != 3:
        return False, ["qft3() should return a QuantumCircuit with 3 qubits."]
    if any(i.name in ("measure", "reset") for i in qc.data):
        return False, ["qft3() should contain gates only, no measurements."]
    m = Operator(qc).data
    if np.allclose(m, dft_matrix(3), atol=1e-8):
        return True, ["qft3(): passed (equal to the 8 x 8 DFT matrix)."]
    return False, ["qft3() does not equal the DFT matrix: " + _diagnose_qft(m, 3)]


def _test_qft(qft):
    for n in range(1, 6):
        try:
            qc = qft(n)
        except NotImplementedError:
            return False, ["qft(n) is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"qft({n}) raised {type(e).__name__}: {e}"]
        if not isinstance(qc, QuantumCircuit) or qc.num_qubits != n:
            return False, [f"qft({n}) should return a QuantumCircuit with {n} qubits."]
        if any(i.name in ("measure", "reset") for i in qc.data):
            return False, [f"qft({n}) should contain gates only, no measurements."]
        m = Operator(qc).data
        if not np.allclose(m, dft_matrix(n), atol=1e-8):
            return False, [f"qft({n}) does not equal the {2 ** n} x {2 ** n} DFT matrix: " + _diagnose_qft(m, n)]
    return True, ["qft(n): passed (n = 1 to 5, each equal to the DFT matrix)."]


CASES = [(1 / 8, 3), (1 / 3, 4), (0.7, 5), (0.05, 3), (5 / 16, 4), (0.9, 2), (0.432, 5)]


def _test_phase_estimation(phase_estimation):
    worst = 0.0
    for phase, t in CASES:
        try:
            qc = phase_estimation(phase, t)
        except NotImplementedError:
            return False, ["phase_estimation() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"phase_estimation({phase:.4g}, {t}) raised {type(e).__name__}: {e}"]
        if not isinstance(qc, QuantumCircuit) or qc.num_qubits != t + 1 or qc.num_clbits != t:
            return False, [f"phase_estimation(phase, {t}) should return a circuit with {t + 1} qubits and {t} classical bits."]
        meas = [i for i in qc.data if i.name == "measure"]
        if sorted(i.qubits[0] for i in meas) != list(range(t)) or any(i.qubits[0] != i.clbits[0] for i in meas):
            return False, ["Measure each counting qubit k (0 to t - 1) into classical bit k, and do not measure the target qubit."]
        got = exact_distribution(qc)
        ref = exact_distribution(reference_phase_estimation(phase, t))
        keys = set(got) | set(ref)
        d = 0.5 * sum(abs(got.get(k, 0) - ref.get(k, 0)) for k in keys)
        worst = max(worst, d)
        if d > 1e-6:
            best = max(got, key=got.get) if got else "none"
            hint = ""
            if set(got) == {"0" * t}:
                hint = " Every result is 0: is the target qubit in |1> (an X gate first), and are the counting qubits in superposition?"
            elif best == format((-round(phase * 2 ** t)) % 2 ** t, f"0{t}b"):
                hint = " The most likely result is the negative of the phase: use the inverse QFT, qft(t).inverse()."
            return False, [f"For phase {phase:.4g} with {t} counting qubits, the results do not match the expected "
                           f"distribution (most likely result {best}, expected "
                           f"{format(round(phase * 2 ** t) % 2 ** t, f'0{t}b')}).{hint} Check that counting qubit k "
                           f"controls the phase gate raised to the power 2^k."]
    return True, [f"phase_estimation(): passed ({len(CASES)} phases, 2 to 5 counting qubits; each distribution exact)."]


def check_l2_module1(qft3, qft, phase_estimation, PHI, SEED):
    try:
        phi, seed = float(PHI), int(SEED)
    except (TypeError, ValueError):
        return False, ["PHI and SEED must be the numbers shown in your Canvas lab check."], None
    if not (0.05 <= phi <= 0.95) or not (100 <= seed <= 999):
        return False, ["PHI and SEED are outside the range Canvas uses. Copy them again from the lab check."], None
    msgs, ok = [], True
    for test, fn in ((_test_qft3, qft3), (_test_qft, qft), (_test_phase_estimation, phase_estimation)):
        good, m = test(fn)
        msgs += m
        ok = ok and good
    if not ok:
        return False, msgs, None
    return True, msgs, personal_value(phi, seed)


passed, messages, value = check_l2_module1(qft3, qft, phase_estimation, PHI, SEED)
for m in messages:
    print(m)
if passed:
    print("\nAll tests passed. Your verification value is", value)
else:
    print("\nNot yet: fix the item above and run this cell again.")

## What you should notice

- A unitary gate's eigenvalues are e^(2πiφ); the eigenphase φ is what phase estimation measures.
- The QFT is the DFT matrix built from about n²/2 gates: H gates, controlled phases π/2^(j − k), and swaps at the end.
- Phase estimation kicks the phase back onto the counting qubits, then the inverse QFT turns that phase pattern into a binary number m with φ ≈ m / 2ᵗ.
- A phase with an exact t-bit expansion (1/8) is found in every shot; any other (1/3) gives the closest t-bit fraction with probability at least 4/π², and more counting qubits make the estimate finer.

Module 3 uses phase estimation to find the period in Shor's algorithm.

Developed through the Intel Semiconductor Education Program at Central State University (ISEP-CSU). CC BY 4.0. Questions: mhadizadeh@centralstate.edu